# Train a Multi-View conv encoder from scratch (Conv-VAE encoder architecture)

We create a sensor processing model that encodes several camera views with a small convolutional encoder (the Conv-VAE encoder architecture) trained **from scratch**, fuses the per-view latents with a configurable fusion head, and trains everything on proprioception.

**Note:** unlike the single-view Conv-VAE this model has no decoder and no KL term; it is trained supervised exactly like the multi-view CNN and ViT models. Keep that in mind when comparing it against the unsupervised single-view Conv-VAE.

The sensor processing object associated with the trained model is in `sensorprocessing/sp_conv_vae_multiview.py`; the fusion heads are in `sensorprocessing/multiview_fusion.py`.

In [ ]:
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from demonstration.demonstration import Demonstration
import training_harness as helper_training
import sensorprocessing.helper_training_data as helper_training_data
from sensorprocessing.sp_conv_vae_multiview import MultiViewConvVAESensorProcessing

device = Config().runtime["device"]
print(f"Using device: {device}")

### Exp-run initialization
Create the exp/run-s that describe the parameters of the training. Some of the code here is structured in such a way as to make the notebook automatizable with papermill.

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# creation_style = "discard-old"
creation_style = "exist-ok"

experiment = "sensorprocessing_conv_vae_multiview"
run = "conv_multiview_concat_proj_128"
# run = "conv_multiview_concat_proj_128"
# run = "conv_multiview_attention_128"
# run = "conv_multiview_gated_256"

# If not None, set the epochs to something different than the exp
epochs = None

# If not None, set an external experiment path (set by the Flow notebooks)
expruns_path = None

# If not None, set an output path (set by the Flow notebooks)
results_path = None

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path).expanduser()
    expruns_path.mkdir(parents=True, exist_ok=True)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment("sensorprocessing_conv_vae_multiview")
    Config().copy_experiment("robot_al5d")
    Config().copy_experiment("demonstration")

if results_path:
    results_path = pathlib.Path(results_path).expanduser()
    results_path.mkdir(parents=True, exist_ok=True)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
exp_robot = Config().get_experiment(exp["robot_exp"], exp["robot_run"])

data_dir = pathlib.Path(exp["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {data_dir}")
print(f"Views: {exp.get('num_views', 2)}  cameras: {exp.get('cameras')}  fusion: {exp.get('fusion_type', 'concat_proj')}  latent: {exp['latent_size']}")

torch.manual_seed(exp.get("seed", 777))

### Create regression training data (multi-view images to proprioception)
Each example is an ordered list of camera images (one per view, in the camera order of the training-data entry) and the normalized robot position. The tensors are cached in the data directory by `helper_training_data`; delete the cache files to rebuild them.

In [ ]:
modelfile = pathlib.Path(exp["data_dir"], exp["proprioception_mlp_model_file"])
will_load_existing = helper_training.model_available(exp) and exp.get("reload_existing_model", True)
if will_load_existing:
    print(f"*** {experiment}/{run} ***: NOT training; model already exists at {modelfile}")

# The data is loaded even when the model exists, it is needed by the tests below
# (and it is cached, so this is cheap after the first run).
tr = helper_training_data.load_multiview_images_as_proprioception_training(exp, exp_robot)
view_inputs_training = tr["view_inputs_training"]
targets_training = tr["targets_training"]
view_inputs_validation = tr["view_inputs_validation"]
targets_validation = tr["targets_validation"]

batch_size = exp.get("batch_size", 32)
# drop_last is on for the training loader: the fusion heads use BatchNorm1d,
# which cannot train on a batch of one sample.
train_loader, test_loader = helper_training_data.make_multiview_loaders(tr, batch_size)

print(f"Training samples: {len(targets_training)}   validation samples: {len(targets_validation)}")
print(f"Views per sample: {len(view_inputs_training)}   image shape: {tuple(view_inputs_training[0].shape[1:])}")

### Create the multi-view model with proprioception regression

In [ ]:
sp = MultiViewConvVAESensorProcessing(exp)
model = sp.enc  # the encoder module is what we train

loss_type = exp.get("loss", "MSELoss")
if loss_type in ("MSELoss", "MSE"):
    criterion = nn.MSELoss()
elif loss_type in ("L1Loss", "L1"):
    criterion = nn.L1Loss()
else:
    raise ValueError(f"Unknown loss {loss_type}")

# Only the trainable parameters go to the optimizer (the backbone may be frozen).
optimizer = optim.Adam(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=exp.get("learning_rate", 0.001),
    weight_decay=exp.get("weight_decay", 0.0001),
)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=exp.get("lr_patience", 10)
)

### Perform the training
The shared training harness handles checkpoints, resuming, best-model selection, early stopping (`early_stopping_patience` in the exp/run) and the final model file.

In [ ]:
model_training_step, model_eval_step = helper_training.make_epoch_steps(
    criterion, train_loader, test_loader, grad_clip_norm=exp.get("grad_clip_norm")
)

epochs = epochs if epochs is not None else exp.get("epochs", 300)

exp.start_timer("train")
model = helper_training.load_or_train(
    exp,
    model,
    optimizer,
    model_training_step,
    model_eval_step,
    epochs=epochs,
    scheduler=lr_scheduler,
)
exp.end_timer("train", verbose=True)

### Test the trained model

In [ ]:
# Recreate the sensor processing object so that it loads the trained weights from disk,
# exactly as the visual-proprioception training and the robot runtime will do.
sp = MultiViewConvVAESensorProcessing(exp)

# 1. The Demonstration-backed inference API (this is what VP training uses)
dataset = exp.get("validation_data") or exp["training_data"]
demo_run, demo_name, cameras = dataset[0]
demo = Demonstration(Config().get_experiment("demonstration", demo_run), demo_name)
print(f"process_demonstration on {demo_name} with cameras {cameras}:")
for timestep in range(min(3, demo.metadata["maxsteps"])):
    latent = sp.process_demonstration(demo, timestep, cameras)
    assert latent.shape[-1] == exp["latent_size"], latent.shape
    print(f"  timestep {timestep}: latent shape {latent.shape}")

# 2. Per-DOF validation error of the full model (encoder + proprioceptor)
model = sp.enc
model.eval()
predictions, targets = [], []
with torch.no_grad():
    for batch_views, batch_targets in test_loader:
        batch_views = helper_training.move_batch_to_device(batch_views, device)
        predictions.append(model(batch_views).cpu())
        targets.append(batch_targets)
predictions = torch.cat(predictions).numpy()
targets = torch.cat(targets).numpy()
field_names = ["height", "distance", "heading", "wrist_angle", "wrist_rotation", "gripper"]
print("\nPer-DOF validation RMSE / MAE (normalized units):")
for index, name in enumerate(field_names[: predictions.shape[1]]):
    rmse = np.sqrt(np.mean((predictions[:, index] - targets[:, index]) ** 2))
    mae = np.mean(np.abs(predictions[:, index] - targets[:, index]))
    print(f"  {name:<16} RMSE {rmse:.4f}   MAE {mae:.4f}")
print(f"Overall RMSE {np.sqrt(np.mean((predictions - targets) ** 2)):.4f}")

# 3. Which view does the model rely on? (weighted_sum / gated heads only)
with torch.no_grad():
    sample_views = [view[:8].to(device) for view in view_inputs_validation]
    features = model.extract_features(sample_views)
    scores = model.fusion.view_scores(features)
if scores is not None:
    print(f"\nMean view weights over 8 validation samples: {scores.mean(dim=0).cpu().numpy().round(3)}")

### Verify the model's encoding and forward methods

In [ ]:
model.eval()
with torch.no_grad():
    sample_views = [view[0].unsqueeze(0).to(device) for view in view_inputs_validation]
    latent = model.encode_views(sample_views)
    position = model(sample_views)
assert latent.shape[1] == exp["latent_size"], f"Expected latent size {exp['latent_size']}, got {latent.shape[1]}"
assert position.shape[1] == exp["output_size"], f"Expected output size {exp['output_size']}, got {position.shape[1]}"
print(f"Latent shape {tuple(latent.shape)}, position shape {tuple(position.shape)}: verification successful!")

### Summary

In [ ]:
print("Training complete!")
print(f"Model file: {modelfile}")
print(f"Encoder channels: {exp.get('encoder_channels', [32, 64, 128, 256])}")
print(f"Number of views: {exp.get('num_views', 2)}  cameras: {exp.get('cameras')}")
print(f"Fusion method: {exp.get('fusion_type', 'concat_proj')}")
print(f"Latent space dimension: {exp['latent_size']}")
print(f"Output dimension (robot DOF): {exp['output_size']}")
print(f"Use the MultiViewConvVAESensorProcessing class to load and use this model for inference.")